# Chapter 7 — Ordering: learned rankers and list reranking

*Companion notebook for* **AI Recommender Systems** *(Manning), Chapter 7.* All logic lives in `recsys.fourstage_recsys.ordering`; this notebook loads, calls and displays.

**Run `00_export_upstream.ipynb` first.** It exports the chapter-5 candidates and cross-encoder scores for two sets of rows:

- **training rows**: upstream models fit on each user's first 70% of positives, labels = the next 10%;
- **test rows**: upstream models fit on the first 80% (chapter 5's training set), labels = the last 20% (chapter 5's test set).

Flow: features → baselines (scored order, popularity, a tuned blend) → LambdaMART with the cross-feature ablation → the same trees with a pointwise loss → DCN-v2 → paired bootstrap CIs → SHAP → the learning curve (the traffic argument) → the ordering pass (MMR + genre cap) over a user sample.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from recsys.fourstage_recsys.ordering import (
    FEATURE_COLS, FEATURE_COLS_NO_CROSS, DCN_DENSE_COLS,
    build_genre_matrix, build_feature_frame, attach_labels, attach_upstream_scores,
    feature_history, request_times,
    ColumnScore, ScoredOrderBaseline, fit_blend, evaluate_ranker, paired_bootstrap,
    train_lambdamart, train_pointwise_gbdt, explain_ranker, feature_importance, lift_curve,
    make_movie_index, add_movie_index, fit_scaler, make_loaders, DCNv2, train_dcn, DCNRanker,
    order_stage, candidate_similarity, primary_genre_map, normalize_relevance,
    evaluate_ordering_pass, lambda_sweep,
)

SEED = 7
np.random.seed(SEED); torch.manual_seed(SEED)
ART = project_root / "data" / "processed" / "chapter07"
K = 10

try:
    import mlflow
    mlflow.set_experiment("ch07-ordering")
except Exception:
    mlflow = None

def log_run(name, metrics, **params):
    if mlflow is None:
        return
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.log_metrics({k.replace("@", "_at_"): v for k, v in metrics.items()})

In [ ]:
ratings = pd.read_parquet(ART / "ratings_sample.parquet")
movies = pd.read_parquet(ART / "movies.parquet")
positives = pd.read_parquet(ART / "positives.parquet")
candidates_fit = pd.read_parquet(ART / "candidates_fit.parquet")
candidates_test = pd.read_parquet(ART / "candidates_test.parquet")

mid = positives[positives["slice"] == "mid"]
late = positives[positives["slice"] == "late"]
genre_matrix = build_genre_matrix(movies)

## Feature frames

Each set of rows gets features computed from the history that precedes its labels, and recency measured from the moment the list is served (the user's first held-out positive):

- training rows: history excludes *mid* and *late*; labels are *mid*;
- test rows: history excludes *late*; labels are *late*.

`n_relevant_test` holds each user's full held-out count so NDCG is normalized exactly as in chapter 5, and the scored-order row reproduces chapter 5's number.

In [ ]:
def make_frame(candidates, labels, future):
    history = feature_history(ratings, future)
    rows = attach_upstream_scores(attach_labels(candidates[["userId", "movieId"]], labels),
                                  candidates)
    return build_feature_frame(rows, history, genre_matrix,
                               request_ts=request_times(future))

fit_frame = make_frame(candidates_fit, mid, pd.concat([mid, late]))
test_frame = make_frame(candidates_test, late, late)
n_relevant_test = late.groupby("userId").size().to_dict()

assert not fit_frame[FEATURE_COLS].isna().any().any()
assert not test_frame[FEATURE_COLS].isna().any().any()
print(f"training rows: {len(fit_frame):,} ({fit_frame.userId.nunique():,} users, "
      f"{fit_frame.relevant.mean():.2%} relevant)")
print(f"test rows:     {len(test_frame):,} ({test_frame.userId.nunique():,} users, "
      f"{test_frame.relevant.mean():.2%} relevant)")

## Baselines

The **scored order** is the row every model has to beat. Two cheap rows keep the comparison honest:

- **popularity order**: the same candidates sorted by `item_log_pop`. If the ranker's lift is mostly popularity, this row shows it.
- **tuned blend**: `cross-encoder score + w · log-popularity`, one weight tuned on the training rows. This is the hand-tuned formula a ranker is supposed to replace; the ranker has to beat it, not just the raw score.

In [ ]:
results, per_user = {}, {}

def run(name, ranker, cols, **params):
    res, pu = evaluate_ranker(test_frame, cols, ranker, k=K,
                              n_relevant=n_relevant_test, return_per_user=True)
    results[name], per_user[name] = res, pu
    log_run(name, res, **params)

run("Scored order (cross-encoder, ch5)", ScoredOrderBaseline(), FEATURE_COLS)
run("Popularity order", ColumnScore("item_log_pop"), FEATURE_COLS)
blend, w = fit_blend(fit_frame, extra_col="item_log_pop")
run(f"Blend: score + {w:.1f}·log-pop", blend, FEATURE_COLS, blend_weight=w)
pd.DataFrame(results).T.round(4)

## LambdaMART, the cross-feature ablation, and the pointwise corner

Three tree models on the same rows: listwise with the hand-built crosses, listwise without them, and pointwise (binary log loss) with them. The last one separates *loss* from *architecture* when DCN-v2 (pointwise) is compared with LambdaMART (listwise).

In [ ]:
lm = train_lambdamart(fit_frame, FEATURE_COLS, seed=SEED)
run("LambdaMART, with cross features", lm, FEATURE_COLS, model="lambdamart")

lm_nox = train_lambdamart(fit_frame, FEATURE_COLS_NO_CROSS, seed=SEED)
run("LambdaMART, no cross features", lm_nox, FEATURE_COLS_NO_CROSS, model="lambdamart")

gbdt_pw = train_pointwise_gbdt(fit_frame, FEATURE_COLS, seed=SEED)
run("GBDT pointwise, with cross features", gbdt_pw, FEATURE_COLS, model="gbdt_binary")
pd.DataFrame(results).T.round(4)

## DCN-v2

Same rows and labels. `DCN_DENSE_COLS` **excludes** the hand-built `x_*` crosses: the cross network is supposed to find them itself. Its fair tree comparison is therefore *LambdaMART, no cross features*. It also gets a learned `movieId` embedding the trees don't have. Validation users are split off the training rows for early stopping.

In [ ]:
movie_index = make_movie_index(pd.concat([fit_frame["movieId"], test_frame["movieId"]]))
fit_idx = add_movie_index(fit_frame, movie_index)
test_idx = add_movie_index(test_frame, movie_index)

users = np.array(sorted(fit_idx["userId"].unique()))
np.random.default_rng(11).shuffle(users)
val_users = set(users[: int(len(users) * 0.15)])
dcn_fit = fit_idx[~fit_idx["userId"].isin(val_users)]
dcn_val = fit_idx[fit_idx["userId"].isin(val_users)]

scaler = fit_scaler(dcn_fit, DCN_DENSE_COLS)            # training rows only
loaders = make_loaders(dcn_fit, dcn_val, DCN_DENSE_COLS, scaler)
dcn_model = DCNv2(n_movies=len(movie_index), dense_dim=len(DCN_DENSE_COLS),
                  emb_dim=32, n_cross=3, mlp_dims=(128, 64))
dcn_model = train_dcn(dcn_model, *loaders, epochs=10, lr=1e-3)
dcn = DCNRanker(dcn_model, scaler, DCN_DENSE_COLS)

res, pu = evaluate_ranker(test_idx, DCN_DENSE_COLS + ["movie_idx"], dcn, k=K,
                          n_relevant=n_relevant_test, return_per_user=True)
results["DCN-v2"], per_user["DCN-v2"] = res, pu
log_run("DCN-v2", res, model="dcn_v2", emb_dim=32, n_cross=3)
table = pd.DataFrame(results).T.round(4)
table                                                    # -> Tables 7.2 and 7.3

## Is the difference real? Paired bootstrap over users

Each row against the scored order, and each model against LambdaMART with crosses. The same users are resampled for both systems in a comparison; an interval that straddles zero means the table cannot tell the two apart.

In [ ]:
base = per_user["Scored order (cross-encoder, ch5)"][f"ndcg@{K}"]
ref = per_user["LambdaMART, with cross features"][f"ndcg@{K}"]
ci_rows = {}
for name, pu in per_user.items():
    vs_base = paired_bootstrap(base, pu[f"ndcg@{K}"])
    vs_lm = paired_bootstrap(ref, pu[f"ndcg@{K}"])
    ci_rows[name] = {
        "Δ vs scored order": vs_base["diff"],
        "95% CI": f"[{vs_base['ci_low']:+.4f}, {vs_base['ci_high']:+.4f}]",
        "Δ vs LambdaMART": vs_lm["diff"],
        "95% CI ": f"[{vs_lm['ci_low']:+.4f}, {vs_lm['ci_high']:+.4f}]",
    }
pd.DataFrame(ci_rows).T.round(4)

## SHAP: what the trees use

Held-out rows only. Attributions are **centered within each user's list**: a ranker's order depends only on differences between one user's candidates, so a user-level feature (constant across the list) can move every score without changing the order. Centering removes that offset, so the beeswarm ranks features by how much they actually reorder lists. Pass `center_by_user=False` to see the raw version for comparison.

In [ ]:
import shap

sample_users = test_frame["userId"].drop_duplicates().sample(
    min(200, test_frame["userId"].nunique()), random_state=SEED)
sample = test_frame[test_frame["userId"].isin(sample_users)]

shap_values = explain_ranker(lm, sample, FEATURE_COLS, center_by_user=True)   # Figure 7.2
shap.dependence_plot("x_genre_affinity", shap_values, sample[FEATURE_COLS],
                     interaction_index="item_log_pop")                        # Figure 7.3
feature_importance(lm, FEATURE_COLS).head(10)

## How much traffic does the ranker need?

LambdaMART trained on growing fractions of the training users, each evaluated on the full test rows, with the lift over the scored order and its bootstrap CI. Where the band clears zero is the point where the ranker starts paying for itself on this data.

In [ ]:
curve = lift_curve(fit_frame, test_frame, FEATURE_COLS,
                   fractions=(0.02, 0.05, 0.1, 0.25, 0.5, 1.0),
                   n_relevant=n_relevant_test, k=K, seed=SEED)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(curve["train_users"], curve["lift"], marker="o", color="black")
ax.fill_between(curve["train_users"], curve["ci_low"], curve["ci_high"],
                color="grey", alpha=0.3, label="95% CI")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xscale("log")
ax.set_xlabel("training users (log scale)")
ax.set_ylabel(f"NDCG@{K} lift over scored order")
ax.legend()
plt.tight_layout(); plt.show()
curve.round(4)

## The ordering pass: MMR + genre cap

The axis changes here: the reranked list should **not** win on NDCG. We report what this stage can honestly self-measure, averaged over a sample of test users: relevance retained (the share of the ranked top-10's normalized relevance the list keeps), ILD@10, the largest genre count in the list, and how many slots changed. Relevance is min-max normalized per list before MMR, so λ means the same trade-off for every user. The cap uses each movie's rarest genre, because MovieLens lists genres alphabetically and "first genre" would mostly cap Action and Adventure.

In [ ]:
scored_test = test_frame.assign(ranker_score=lm.predict(test_frame[FEATURE_COLS]))

ordering_table = evaluate_ordering_pass(scored_test, "ranker_score", genre_matrix,
                                        n_users=500, k=K, lam=0.7, cap=3, seed=SEED)
ordering_table.round(3)                                  # -> Table 7.4

In [ ]:
sweep = lambda_sweep(scored_test, "ranker_score", genre_matrix,
                     lams=np.linspace(0.0, 1.0, 11), n_users=500, k=K, seed=SEED)

fig, ax = plt.subplots(figsize=(6, 4))
ax2 = ax.twinx()
l1 = ax.plot(sweep["lambda"], sweep[f"ILD@{K}"], marker="o", color="black", label=f"ILD@{K}")
l2 = ax2.plot(sweep["lambda"], sweep["relevance retained"], marker="s", color="grey",
              linestyle="--", label="relevance retained (right)")
ax.set_xlabel("λ")
ax.set_ylabel(f"ILD@{K}")
ax2.set_ylabel("relevance retained")
ax.legend(l1 + l2, [l.get_label() for l in l1 + l2], loc="lower left")
plt.tight_layout(); plt.show()                           # -> Figure 7.5

In [ ]:
# Figure 7.6: one user's list before and after. Pick a user whose ranked top-10
# actually hits the cap, so there is something to show.
genres = primary_genre_map(genre_matrix)
titles = movies.set_index("movieId")["title"]

def top_k(u):
    return scored_test[scored_test.userId == u].sort_values("ranker_score", ascending=False)

user_id = next(u for u in sorted(scored_test.userId.unique())
               if len(top_k(u)) >= K
               and pd.Series([genres[m] for m in top_k(u).movieId[:K]]).value_counts().max() > 3)
cand = top_k(user_id)
ids = cand["movieId"].tolist()
rel = normalize_relevance(cand["ranker_score"].to_numpy())
after = order_stage(ids, rel, candidate_similarity(ids, genre_matrix), genres,
                    k=K, lam=0.7, cap=3)
pd.DataFrame({
    "before": [f"{titles.get(m, m)}  [{genres[m]}]" for m in ids[:K]],
    "after": [f"{titles.get(m, m)}  [{genres[m]}]" for m in after],
})

## Filling the chapter

- Results table → Table 7.2 (baselines + LambdaMART) and Table 7.3 (all rows). The scored-order row should match chapter 5's cross-encoder NDCG@10 (see the wiring check in `00_export_upstream.ipynb`).
- Bootstrap table → the "is it real" sentences in the evaluation and comparison sections. Don't describe a difference whose CI straddles zero as a win.
- SHAP cell → Figures 7.2 and 7.3; learning curve → the "does it need to be there" section.
- Ordering cells → Table 7.4 (now averaged over 500 users), Figure 7.5 and Figure 7.6.
- Never report an offline NDCG lift for the reranking transforms. By design they trade relevance for diversity.